# 🩺 Multimodal Anemia Screening System
## Conjunctiva Image Analysis with Tabular Data Fusion

This notebook implements an end-to-end anemia severity classification pipeline using:
- **Conjunctiva images** (MobileNetV2 CNN backbone)
- **Tabular patient data** (age, gender)
- **Multimodal fusion** via Keras Functional API
- **Explainability** via SHAP and GradCAM
- **Nutritional recommendation engine**
- **FastAPI deployment** and **TFLite export**

| Class | Count |
|-------|-------|
| Normal | 286 |
| Mild | 144 |
| Moderate | 232 |
| Severe | 48 |


## Section 1 — Environment Setup & Imports

Install dependencies and configure global constants.

In [ ]:
# ── Install required packages ──────────────────────────────────────────────
import subprocess, sys

def pip_install(*pkgs):
    """Install packages via pip if not already importable."""
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip_install(
    "tensorflow>=2.12",
    "scikit-learn>=1.3",
    "xgboost>=2.0",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "shap",
    "fastapi",
    "uvicorn[standard]",
    "Pillow",
    "joblib",
    "tqdm",
)
print("All packages installed.")


In [ ]:
# ── Core imports ───────────────────────────────────────────────────────────
import os, json, warnings, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import joblib
from tqdm.auto import tqdm
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

import shap

warnings.filterwarnings("ignore")
print(f"TensorFlow version: {tf.__version__}")
print(f"NumPy version:      {np.__version__}")
print(f"Pandas version:     {pd.__version__}")


In [ ]:
# ── Constants ──────────────────────────────────────────────────────────────
IMAGE_SIZE    = (224, 224)
BATCH_SIZE    = 32
NUM_CLASSES   = 4
RANDOM_SEED   = 42
CLASS_NAMES   = ["Normal", "Mild", "Moderate", "Severe"]
LABEL_MAP     = {"Normal": 0, "Mild": 1, "Moderate": 2, "Severe": 3}

# Resolve the directory in which this notebook lives.
# When run from `project/`, BASE_DIR == the project root.
BASE_DIR       = os.path.dirname(os.path.abspath("__file__"))
DATA_DIR       = os.path.join(BASE_DIR, "data")
MODEL_DIR      = os.path.join(BASE_DIR, "models", "saved_models")
NUTRITION_PATH = os.path.join(BASE_DIR, "nutrition", "nutrition_rules.json")
METADATA_PATH  = os.path.join(DATA_DIR, "metadata.csv")

# ── Reproducibility ────────────────────────────────────────────────────────
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)

# ── Directory creation ─────────────────────────────────────────────────────
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(os.path.join(BASE_DIR, "models", "tflite"), exist_ok=True)

print(f"BASE_DIR       : {BASE_DIR}")
print(f"DATA_DIR       : {DATA_DIR}")
print(f"MODEL_DIR      : {MODEL_DIR}")
print(f"NUTRITION_PATH : {NUTRITION_PATH}")
print("Directories ready ✓")


## Section 2 — Data Exploration

Load the metadata CSV, inspect quality, and visualise class and demographic distributions.

In [ ]:
# ── Load metadata ──────────────────────────────────────────────────────────
df = pd.read_csv(METADATA_PATH)
print(f"Shape: {df.shape}")
print(df.head())
print("\nDtypes:")
print(df.dtypes)


In [ ]:
# ── Quality checks ─────────────────────────────────────────────────────────
print("=== Missing values ===")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")

# Verify all image files exist
missing_images = [
    p for p in df["image_path"]
    if not os.path.exists(os.path.join(BASE_DIR, p))
]
print(f"\nMissing image files: {len(missing_images)}")
if missing_images:
    print("  First 5:", missing_images[:5])


In [ ]:
# ── Class distribution ─────────────────────────────────────────────────────
dist = df["diagnosis"].value_counts()
print("=== Class distribution ===")
print(dist.to_string())
print(f"\nImbalance ratio (majority/minority): {dist.max()/dist.min():.1f}x")


In [ ]:
# ── Sample images per class ────────────────────────────────────────────────
fig, axes = plt.subplots(
    len(CLASS_NAMES), 5,
    figsize=(15, 3 * len(CLASS_NAMES))
)
fig.suptitle("Sample Conjunctiva Images per Class", fontsize=16, fontweight="bold")

for row_idx, cls in enumerate(CLASS_NAMES):
    samples = df[df["diagnosis"] == cls].sample(
        min(5, (df["diagnosis"] == cls).sum()),
        random_state=RANDOM_SEED
    )
    for col_idx, (_, sample) in enumerate(samples.iterrows()):
        img_path = os.path.join(BASE_DIR, sample["image_path"])
        img = Image.open(img_path).convert("RGB")
        axes[row_idx, col_idx].imshow(img)
        axes[row_idx, col_idx].set_title(
            f"{cls}\nAge {sample['age']} {sample['gender'][0]}",
            fontsize=8
        )
        axes[row_idx, col_idx].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "models", "sample_images.png"), dpi=100)
plt.show()


In [ ]:
# ── Distribution charts ────────────────────────────────────────────────────
PALETTE = {
    "Normal": "#27ae60",
    "Mild": "#f39c12",
    "Moderate": "#e67e22",
    "Severe": "#e74c3c"
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Class frequency bar chart
class_counts = df["diagnosis"].value_counts().reindex(CLASS_NAMES)
bars = axes[0].bar(CLASS_NAMES, class_counts.values,
                   color=[PALETTE[c] for c in CLASS_NAMES], edgecolor="black")
for bar, val in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
                 str(val), ha="center", fontweight="bold")
axes[0].set_title("Class Frequency", fontweight="bold")
axes[0].set_xlabel("Diagnosis")
axes[0].set_ylabel("Count")
axes[0].set_ylim(0, class_counts.max() * 1.15)

# 2. Age histogram coloured by class
for cls in CLASS_NAMES:
    subset = df[df["diagnosis"] == cls]["age"]
    axes[1].hist(subset, bins=20, alpha=0.6, label=cls, color=PALETTE[cls], edgecolor="white")
axes[1].set_title("Age Distribution by Class", fontweight="bold")
axes[1].set_xlabel("Age (years)")
axes[1].set_ylabel("Frequency")
axes[1].legend()

# 3. Stacked bar: gender split per class
gender_pivot = (
    df.groupby(["diagnosis", "gender"])
      .size()
      .unstack(fill_value=0)
      .reindex(CLASS_NAMES)
)
gender_pivot.plot(
    kind="bar", stacked=True,
    ax=axes[2], color=["#3498db", "#e91e8c"], edgecolor="white"
)
axes[2].set_title("Gender Split per Class", fontweight="bold")
axes[2].set_xlabel("Diagnosis")
axes[2].set_ylabel("Count")
axes[2].legend(title="Gender")
axes[2].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "models", "eda_plots.png"), dpi=100)
plt.show()
print("EDA plots saved.")


## Section 3 — Image Preprocessing & Augmentation

Define image loading helpers and a `tf.data` pipeline with online augmentation for training.

In [ ]:
# ── Image loading ──────────────────────────────────────────────────────────
def load_and_preprocess_image(path: str) -> tf.Tensor:
    """
    Load a PNG/JPEG image from *path*, resize to IMAGE_SIZE, and normalise
    pixel values to the range [0, 1].

    Args:
        path: Absolute or relative path to the image file.

    Returns:
        Float32 tensor of shape (H, W, 3) with values in [0, 1].
    """
    raw   = tf.io.read_file(path)
    image = tf.image.decode_png(raw, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image


# Quick smoke-test
sample_path = os.path.join(BASE_DIR, df["image_path"].iloc[0])
test_img = load_and_preprocess_image(sample_path)
print(f"Loaded image tensor shape: {test_img.shape}, dtype: {test_img.dtype}")
print(f"Pixel range: [{test_img.numpy().min():.3f}, {test_img.numpy().max():.3f}]")


In [ ]:
# ── Augmentation layer ─────────────────────────────────────────────────────
def build_augmentation_pipeline() -> keras.Sequential:
    """
    Build a Keras Sequential augmentation pipeline for training.

    Augmentations applied (all deterministic-seed compatible):
        - Random horizontal flip
        - Random rotation ±10 %
        - Random zoom ±10 %
        - Random brightness ±15 %
        - Random contrast factor 0.8–1.2

    Returns:
        A compiled keras.Sequential model used as a preprocessing layer.
    """
    return keras.Sequential(
        [
            layers.RandomFlip("horizontal", seed=RANDOM_SEED),
            layers.RandomRotation(0.10, seed=RANDOM_SEED),
            layers.RandomZoom(0.10, seed=RANDOM_SEED),
            layers.RandomBrightness(0.15, seed=RANDOM_SEED),
            layers.RandomContrast(0.2, seed=RANDOM_SEED),
        ],
        name="augmentation_pipeline",
    )


augmentor = build_augmentation_pipeline()
print("Augmentation pipeline:")
augmentor.summary()


In [ ]:
# ── tf.data dataset builder ────────────────────────────────────────────────
def build_tf_dataset(
    sub_df: pd.DataFrame,
    augment: bool = False,
    batch_size: int = BATCH_SIZE,
    shuffle: bool = True,
) -> tf.data.Dataset:
    """
    Build a batched, prefetched tf.data.Dataset from a metadata DataFrame.

    Args:
        sub_df:     DataFrame subset with columns ['image_path', 'age_scaled',
                    'gender_encoded', 'label'].
        augment:    If True, apply random augmentation (training only).
        batch_size: Number of samples per batch.
        shuffle:    If True, shuffle the dataset.

    Returns:
        A tf.data.Dataset yielding ((image, tabular), label) tuples.
    """
    paths    = sub_df["image_path"].apply(
                   lambda p: os.path.join(BASE_DIR, p)).tolist()
    tabular  = sub_df[["age_scaled", "gender_encoded"]].values.astype(np.float32)
    labels   = sub_df["label"].values.astype(np.int32)

    def _load(path, tab, label):
        """Load and optionally augment a single sample."""
        img = load_and_preprocess_image(path)
        if augment:
            img = augmentor(tf.expand_dims(img, 0), training=True)[0]
        label_oh = tf.one_hot(label, NUM_CLASSES)
        return (img, tab), label_oh

    ds = tf.data.Dataset.from_tensor_slices((paths, tabular, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(sub_df), seed=RANDOM_SEED)
    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


print("build_tf_dataset() defined ✓")


In [ ]:
# ── Visualise augmented samples ────────────────────────────────────────────
sample_df = df.sample(8, random_state=RANDOM_SEED).reset_index(drop=True)

fig, axes = plt.subplots(2, 8, figsize=(20, 5))
fig.suptitle("Original vs. Augmented Images", fontsize=14, fontweight="bold")

for i, (_, row) in enumerate(sample_df.iterrows()):
    img_path = os.path.join(BASE_DIR, row["image_path"])
    orig = load_and_preprocess_image(img_path).numpy()

    aug_input = tf.expand_dims(
        tf.cast(orig, tf.float32), 0
    )
    aug  = augmentor(aug_input, training=True)[0].numpy()
    aug  = np.clip(aug, 0, 1)

    axes[0, i].imshow(orig)
    axes[0, i].set_title(row["diagnosis"], fontsize=8)
    axes[0, i].axis("off")

    axes[1, i].imshow(aug)
    axes[1, i].set_title("augmented", fontsize=8)
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=10, fontweight="bold")
axes[1, 0].set_ylabel("Augmented", fontsize=10, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "models", "augmentation_samples.png"), dpi=100)
plt.show()


## Section 4 — Tabular Feature Processing

Encode categorical features, scale numeric features, and produce a stratified 70/15/15 train/validation/test split.

In [ ]:
# ── Encode labels & features ───────────────────────────────────────────────
# Map diagnosis to integer label
df["label"] = df["diagnosis"].map(LABEL_MAP)

# Encode gender: Male=1, Female=0
gender_encoder = LabelEncoder()
df["gender_encoded"] = gender_encoder.fit_transform(df["gender"])
print("Gender classes:", dict(zip(gender_encoder.classes_,
                                  gender_encoder.transform(gender_encoder.classes_))))

# Scale age
age_scaler = StandardScaler()
df["age_scaled"] = age_scaler.fit_transform(df[["age"]])
print(f"Age mean: {age_scaler.mean_[0]:.2f}, std: {age_scaler.scale_[0]:.2f}")


In [ ]:
# ── Stratified 70 / 15 / 15 split ─────────────────────────────────────────
# First split: train (70%) vs temp (30%)
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=RANDOM_SEED,
)
# Second split: val (15%) vs test (15%) from the temp 30%
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=RANDOM_SEED,
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train : {len(train_df):>4} samples")
print(f"Val   : {len(val_df):>4} samples")
print(f"Test  : {len(test_df):>4} samples")
print(f"Total : {len(train_df)+len(val_df)+len(test_df):>4} samples")
print()
print("Class distribution in splits:")
for split_name, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    counts = split["diagnosis"].value_counts().reindex(CLASS_NAMES).fillna(0).astype(int)
    print(f"  {split_name}: { {k:v for k,v in counts.items()} }")


In [ ]:
# ── Persist encoder and scaler ─────────────────────────────────────────────
joblib.dump(gender_encoder, os.path.join(MODEL_DIR, "label_encoder.pkl"))
joblib.dump(age_scaler,     os.path.join(MODEL_DIR, "scaler.pkl"))
print("Saved: label_encoder.pkl")
print("Saved: scaler.pkl")


## Section 5 — Tabular Baseline Models

Train Logistic Regression, Random Forest, and XGBoost on the two tabular features (age, gender). These baselines establish a lower bound for performance.

In [ ]:
# ── Prepare tabular arrays ─────────────────────────────────────────────────
FEAT_COLS = ["age_scaled", "gender_encoded"]

X_train_tab = train_df[FEAT_COLS].values
y_train_tab = train_df["label"].values

X_val_tab   = val_df[FEAT_COLS].values
y_val_tab   = val_df["label"].values

X_test_tab  = test_df[FEAT_COLS].values
y_test_tab  = test_df["label"].values

print(f"Tabular feature shape (train): {X_train_tab.shape}")


In [ ]:
# ── Train & evaluate tabular models ───────────────────────────────────────
tabular_models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=200, class_weight="balanced",
        n_jobs=-1, random_state=RANDOM_SEED
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=4,
        use_label_encoder=False, eval_metric="mlogloss",
        random_state=RANDOM_SEED, verbosity=0,
    ),
}

tab_results = {}
for name, clf in tabular_models.items():
    clf.fit(X_train_tab, y_train_tab)
    y_pred = clf.predict(X_test_tab)
    acc = accuracy_score(y_test_tab, y_pred)
    f1  = f1_score(y_test_tab, y_pred, average="macro", zero_division=0)
    tab_results[name] = {"accuracy": acc, "macro_f1": f1, "model": clf, "preds": y_pred}
    print(f"{name:25s}  acc={acc:.3f}  macro-F1={f1:.3f}")

# Best tabular model by macro F1
best_tab_name = max(tab_results, key=lambda k: tab_results[k]["macro_f1"])
best_tab_model = tab_results[best_tab_name]["model"]
print(f"\nBest tabular model: {best_tab_name}")


In [ ]:
# ── Confusion matrices & Severe recall ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Tabular Baseline — Confusion Matrices (Test Set)", fontweight="bold")

for ax, (name, res) in zip(axes, tab_results.items()):
    cm_vals = confusion_matrix(y_test_tab, res["preds"])
    disp = ConfusionMatrixDisplay(cm_vals, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "models", "tabular_confusion_matrices.png"), dpi=100)
plt.show()

# Per-class recall
print("\n=== Per-class Recall (Severe highlighted if < 0.80) ===")
for name, res in tab_results.items():
    report = classification_report(
        y_test_tab, res["preds"],
        target_names=CLASS_NAMES, output_dict=True, zero_division=0
    )
    severe_recall = report["Severe"]["recall"]
    flag = " \033[91m⚠ Severe recall < 0.80\033[0m" if severe_recall < 0.80 else " ✓"
    print(f"  {name:25s}  Severe recall={severe_recall:.3f}{flag}")


In [ ]:
# ── Save best tabular model ────────────────────────────────────────────────
best_tab_path = os.path.join(MODEL_DIR, "best_tabular_model.pkl")
joblib.dump(best_tab_model, best_tab_path)
print(f"Saved best tabular model ({best_tab_name}) → {best_tab_path}")


## Section 6 — Visual CNN Model

Fine-tune **MobileNetV2** on the conjunctiva images in two phases:
1. **Phase 1** – Freeze the base; train the classification head for 10 epochs.
2. **Phase 2** – Unfreeze the last 30 layers and fine-tune with a lower LR.

In [ ]:
# ── Compute class weights from training split ──────────────────────────────
from sklearn.utils.class_weight import compute_class_weight

class_weights_arr = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_df["label"].values,
)
CLASS_WEIGHTS = {i: w for i, w in enumerate(class_weights_arr)}
print("Class weights:", {CLASS_NAMES[k]: round(v, 3) for k, v in CLASS_WEIGHTS.items()})


In [ ]:
# ── Build CNN-only dataset (images + one-hot labels, no tabular branch) ───
def build_image_only_dataset(sub_df, augment=False, batch_size=BATCH_SIZE, shuffle=True):
    """
    Build a tf.data.Dataset that yields (image_tensor, one_hot_label) pairs.

    Args:
        sub_df:     DataFrame with 'image_path' and 'label' columns.
        augment:    Apply augmentation if True.
        batch_size: Batch size.
        shuffle:    Shuffle the dataset.

    Returns:
        Batched, prefetched tf.data.Dataset.
    """
    paths  = sub_df["image_path"].apply(
                 lambda p: os.path.join(BASE_DIR, p)).tolist()
    labels = sub_df["label"].values.astype(np.int32)

    def _load(path, label):
        img = load_and_preprocess_image(path)
        if augment:
            img = augmentor(tf.expand_dims(img, 0), training=True)[0]
        return img, tf.one_hot(label, NUM_CLASSES)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(sub_df), seed=RANDOM_SEED)
    return ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_img_ds = build_image_only_dataset(train_df, augment=True)
val_img_ds   = build_image_only_dataset(val_df,   augment=False, shuffle=False)
test_img_ds  = build_image_only_dataset(test_df,  augment=False, shuffle=False)
print("Image-only datasets ready ✓")


In [ ]:
# ── Build the visual CNN model ─────────────────────────────────────────────
def build_visual_model() -> Model:
    """
    Construct a MobileNetV2-based image classification model.

    Architecture:
        Input (224×224×3)
          → MobileNetV2 base (pre-trained on ImageNet, base frozen initially)
          → GlobalAveragePooling2D
          → Dense(256, relu) + BatchNorm + Dropout(0.4)
          → Dense(NUM_CLASSES, softmax)

    Returns:
        Compiled Keras Model.
    """
    base = MobileNetV2(
        input_shape=(*IMAGE_SIZE, 3),
        include_top=False,
        weights="imagenet",
    )
    base.trainable = False  # Phase 1: frozen

    inputs = keras.Input(shape=(*IMAGE_SIZE, 3), name="image_input")
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(256, activation="relu", name="head_dense")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)

    model = Model(inputs, outputs, name="visual_cnn")
    model.compile(
        optimizer=Adam(1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model, base


visual_model, mobilenet_base = build_visual_model()
visual_model.summary()


In [ ]:
# ── Phase 1 Training — frozen base ────────────────────────────────────────
print("=== Phase 1: Training classification head (base frozen) ===")

cnn_phase1_ckpt = os.path.join(MODEL_DIR, "visual_cnn_phase1.h5")

callbacks_p1 = [
    EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
    ModelCheckpoint(cnn_phase1_ckpt, save_best_only=True, monitor="val_accuracy"),
    ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-6),
]

history_p1 = visual_model.fit(
    train_img_ds,
    validation_data=val_img_ds,
    epochs=10,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks_p1,
    verbose=1,
)
print("Phase 1 complete ✓")


In [ ]:
# ── Phase 2 Training — partial unfreeze ───────────────────────────────────
print("=== Phase 2: Fine-tuning last 30 MobileNetV2 layers ===")

# Unfreeze last 30 layers
mobilenet_base.trainable = True
for layer in mobilenet_base.layers[:-30]:
    layer.trainable = False

visual_model.compile(
    optimizer=Adam(1e-4),  # lower LR for fine-tuning
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

cnn_best_ckpt = os.path.join(MODEL_DIR, "visual_cnn_best.h5")

callbacks_p2 = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor="val_accuracy"),
    ModelCheckpoint(cnn_best_ckpt, save_best_only=True, monitor="val_accuracy"),
    ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-7),
]

history_p2 = visual_model.fit(
    train_img_ds,
    validation_data=val_img_ds,
    epochs=20,
    class_weight=CLASS_WEIGHTS,
    callbacks=callbacks_p2,
    verbose=1,
    initial_epoch=len(history_p1.epoch),
)
print("Phase 2 complete ✓")


In [ ]:
# ── Plot training curves ───────────────────────────────────────────────────
def plot_training_history(h1, h2, title="Training Curves", save_path=None):
    """
    Combine two Keras History objects and plot accuracy/loss curves.

    Args:
        h1: History from Phase 1.
        h2: History from Phase 2.
        title: Plot title.
        save_path: Optional path to save the figure.
    """
    acc  = h1.history["accuracy"]    + h2.history["accuracy"]
    vacc = h1.history["val_accuracy"]+ h2.history["val_accuracy"]
    loss = h1.history["loss"]        + h2.history["loss"]
    vloss= h1.history["val_loss"]    + h2.history["val_loss"]

    epochs = range(1, len(acc) + 1)
    phase_boundary = len(h1.epoch)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for ax, train_vals, val_vals, metric in [
        (ax1, acc, vacc, "Accuracy"),
        (ax2, loss, vloss, "Loss"),
    ]:
        ax.plot(epochs, train_vals, "b-o", markersize=3, label="Train")
        ax.plot(epochs, val_vals,   "r-o", markersize=3, label="Val")
        ax.axvline(phase_boundary, ls="--", color="gray", label="Phase 2 start")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(metric)
        ax.set_title(metric)
        ax.legend()
        ax.grid(alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100)
    plt.show()


plot_training_history(
    history_p1, history_p2,
    title="Visual CNN — Training Curves",
    save_path=os.path.join(BASE_DIR, "models", "visual_cnn_training.png"),
)


In [ ]:
# ── Evaluate Visual CNN on test set ───────────────────────────────────────
y_true_vis = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_img_ds])
y_pred_vis = np.argmax(visual_model.predict(test_img_ds, verbose=0), axis=1)

vis_acc = accuracy_score(y_true_vis, y_pred_vis)
vis_f1  = f1_score(y_true_vis, y_pred_vis, average="macro", zero_division=0)
print(f"Visual CNN — Test Accuracy: {vis_acc:.4f}  Macro F1: {vis_f1:.4f}")
print()
print(classification_report(y_true_vis, y_pred_vis, target_names=CLASS_NAMES, zero_division=0))


## Section 7 — Multimodal Fusion Model

Combine the CNN image branch with a tabular MLP branch using the **Keras Functional API**. The fused model processes conjunctiva images and patient demographics simultaneously.

In [ ]:
# ── Build multimodal datasets ──────────────────────────────────────────────
train_mm_ds = build_tf_dataset(train_df, augment=True)
val_mm_ds   = build_tf_dataset(val_df,   augment=False, shuffle=False)
test_mm_ds  = build_tf_dataset(test_df,  augment=False, shuffle=False)
print("Multimodal datasets ready ✓")
print("  A batch yields: (image_batch, tabular_batch), one_hot_label_batch")


In [ ]:
# ── Build multimodal fusion model ─────────────────────────────────────────
def build_multimodal_model() -> Model:
    """
    Construct the multimodal fusion model.

    Architecture:
        Image branch:
            Input (224×224×3)
              → MobileNetV2 (ImageNet weights, trainable=False initially)
              → GlobalAveragePooling2D
              → Dense(256, relu) + BatchNorm + Dropout(0.4)

        Tabular branch:
            Input (2,) [age_scaled, gender_encoded]
              → Dense(32, relu) + BatchNorm
              → Dense(16, relu)

        Fusion:
            Concatenate([image_feat, tab_feat])
              → Dense(128, relu) + Dropout(0.3)
              → Dense(NUM_CLASSES, softmax)

    Returns:
        Compiled Keras Model (both branches, base frozen).
        MobileNetV2 base model (for later unfreezing).
    """
    # ─── Image branch ───
    mm_base = MobileNetV2(
        input_shape=(*IMAGE_SIZE, 3),
        include_top=False,
        weights="imagenet",
    )
    mm_base.trainable = False

    img_input = keras.Input(shape=(*IMAGE_SIZE, 3), name="image_input")
    x_img = mm_base(img_input, training=False)
    x_img = layers.GlobalAveragePooling2D(name="gap")(x_img)
    x_img = layers.Dense(256, activation="relu", name="img_dense")(x_img)
    x_img = layers.BatchNormalization()(x_img)
    x_img = layers.Dropout(0.4)(x_img)

    # ─── Tabular branch ───
    tab_input = keras.Input(shape=(2,), name="tabular_input")
    x_tab = layers.Dense(32, activation="relu", name="tab_dense1")(tab_input)
    x_tab = layers.BatchNormalization()(x_tab)
    x_tab = layers.Dense(16, activation="relu", name="tab_dense2")(x_tab)

    # ─── Fusion ───
    fused = layers.Concatenate(name="fusion")([x_img, x_tab])
    fused = layers.Dense(128, activation="relu", name="fusion_dense")(fused)
    fused = layers.Dropout(0.3)(fused)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(fused)

    model = Model(inputs=[img_input, tab_input], outputs=outputs, name="anemia_multimodal")
    model.compile(
        optimizer=Adam(1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model, mm_base


mm_model, mm_base = build_multimodal_model()
mm_model.summary()


In [ ]:
# ── Multimodal Phase 1 ─────────────────────────────────────────────────────
print("=== Multimodal Phase 1: frozen base ===")

mm_ckpt_p1 = os.path.join(MODEL_DIR, "anemia_multimodal_phase1.h5")
mm_callbacks_p1 = [
    EarlyStopping(patience=4, restore_best_weights=True, monitor="val_accuracy"),
    ModelCheckpoint(mm_ckpt_p1, save_best_only=True, monitor="val_accuracy"),
    ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-6),
]

mm_history_p1 = mm_model.fit(
    train_mm_ds,
    validation_data=val_mm_ds,
    epochs=10,
    class_weight=CLASS_WEIGHTS,
    callbacks=mm_callbacks_p1,
    verbose=1,
)
print("Multimodal Phase 1 complete ✓")


In [ ]:
# ── Multimodal Phase 2 ─────────────────────────────────────────────────────
print("=== Multimodal Phase 2: fine-tuning last 30 base layers ===")

mm_base.trainable = True
for layer in mm_base.layers[:-30]:
    layer.trainable = False

mm_model.compile(
    optimizer=Adam(1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

mm_best_ckpt = os.path.join(MODEL_DIR, "anemia_multimodal.h5")
mm_callbacks_p2 = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor="val_accuracy"),
    ModelCheckpoint(mm_best_ckpt, save_best_only=True, monitor="val_accuracy"),
    ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-7),
]

mm_history_p2 = mm_model.fit(
    train_mm_ds,
    validation_data=val_mm_ds,
    epochs=20,
    class_weight=CLASS_WEIGHTS,
    callbacks=mm_callbacks_p2,
    verbose=1,
    initial_epoch=len(mm_history_p1.epoch),
)
print("Multimodal Phase 2 complete ✓")


In [ ]:
# ── Compare multimodal vs visual-only ─────────────────────────────────────
y_true_mm = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_mm_ds])
y_pred_mm = np.argmax(mm_model.predict(test_mm_ds, verbose=0), axis=1)

mm_acc = accuracy_score(y_true_mm, y_pred_mm)
mm_f1  = f1_score(y_true_mm, y_pred_mm, average="macro", zero_division=0)

print("=== Model Comparison (Test Set) ===")
print(f"{'Model':<25} {'Accuracy':>10} {'Macro F1':>10}")
print("-" * 48)
print(f"{'Visual CNN only':<25} {vis_acc:>10.4f} {vis_f1:>10.4f}")
print(f"{'Multimodal Fusion':<25} {mm_acc:>10.4f} {mm_f1:>10.4f}")

delta_f1 = mm_f1 - vis_f1
print(f"\nMultimodal improvement: Δ Macro F1 = {delta_f1:+.4f}")

plot_training_history(
    mm_history_p1, mm_history_p2,
    title="Multimodal Fusion — Training Curves",
    save_path=os.path.join(BASE_DIR, "models", "multimodal_training.png"),
)


## Section 8 — Cross Validation

Use `StratifiedKFold(n_splits=5)` to estimate multimodal model variance. Each fold trains a fresh model for a reduced number of epochs to keep runtime feasible.

In [ ]:
# ── 5-Fold Stratified Cross Validation ────────────────────────────────────
# NOTE: Full training (Sections 6-7) is computationally expensive.
# Cross-validation here trains for a limited 5 epochs per fold on the
# combined train+val set to estimate variance efficiently.

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_df = pd.concat([train_df, val_df]).reset_index(drop=True)

cv_results = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(cv_df, cv_df["label"])):
    print(f"\n─── Fold {fold+1}/5 ───────────────────────────────")
    fold_train = cv_df.iloc[tr_idx]
    fold_val   = cv_df.iloc[va_idx]

    fold_train_ds = build_tf_dataset(fold_train, augment=True)
    fold_val_ds   = build_tf_dataset(fold_val,   augment=False, shuffle=False)

    # Fresh model each fold
    fold_model, _ = build_multimodal_model()

    fold_weights = compute_class_weight(
        "balanced",
        classes=np.arange(NUM_CLASSES),
        y=fold_train["label"].values
    )
    fold_class_w = {i: w for i, w in enumerate(fold_weights)}

    fold_model.fit(
        fold_train_ds,
        validation_data=fold_val_ds,
        epochs=5,
        class_weight=fold_class_w,
        callbacks=[EarlyStopping(patience=2, restore_best_weights=True)],
        verbose=0,
    )

    y_true_cv = np.concatenate(
        [np.argmax(y.numpy(), axis=1) for _, y in fold_val_ds]
    )
    y_pred_cv = np.argmax(fold_model.predict(fold_val_ds, verbose=0), axis=1)

    fold_acc = accuracy_score(y_true_cv, y_pred_cv)
    fold_f1  = f1_score(y_true_cv, y_pred_cv, average="macro", zero_division=0)
    cv_results.append({"fold": fold+1, "accuracy": fold_acc, "macro_f1": fold_f1})
    print(f"  Fold {fold+1}: acc={fold_acc:.4f}  macro-F1={fold_f1:.4f}")

    # Free GPU memory
    del fold_model
    keras.backend.clear_session()

cv_results_df = pd.DataFrame(cv_results)
print("\n=== Cross-Validation Results ===")
print(cv_results_df.to_string(index=False))
print(f"\nMean Accuracy : {cv_results_df['accuracy'].mean():.4f} ± {cv_results_df['accuracy'].std():.4f}")
print(f"Mean Macro F1 : {cv_results_df['macro_f1'].mean():.4f} ± {cv_results_df['macro_f1'].std():.4f}")


## Section 9 — Model Evaluation

Comprehensive evaluation of the final multimodal model on the held-out test set, including per-class metrics and flagging critical Severe recall.

In [ ]:
# ── Full classification report ─────────────────────────────────────────────
# Reload best checkpoint to ensure we use the best weights
mm_model = keras.models.load_model(mm_best_ckpt)

y_true_final = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_mm_ds])
y_pred_final = np.argmax(mm_model.predict(test_mm_ds, verbose=0), axis=1)
y_prob_final = mm_model.predict(test_mm_ds, verbose=0)

print("=== Final Classification Report (Multimodal — Test Set) ===\n")
print(classification_report(y_true_final, y_pred_final, target_names=CLASS_NAMES, zero_division=0))


In [ ]:
# ── Confusion matrix heatmap ───────────────────────────────────────────────
cm_final = confusion_matrix(y_true_final, y_pred_final)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
sns.heatmap(
    cm_final, annot=True, fmt="d", cmap="Blues",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    ax=axes[0], linewidths=0.5, linecolor="gray",
)
axes[0].set_xlabel("Predicted", fontweight="bold")
axes[0].set_ylabel("True", fontweight="bold")
axes[0].set_title("Confusion Matrix (Test Set)", fontweight="bold")

# Per-class recall bar chart
report_dict = classification_report(
    y_true_final, y_pred_final,
    target_names=CLASS_NAMES, output_dict=True, zero_division=0
)
recalls = [report_dict[c]["recall"] for c in CLASS_NAMES]
bar_colors = [
    "#e74c3c" if (c == "Severe" and r < 0.80) else PALETTE[c]
    for c, r in zip(CLASS_NAMES, recalls)
]
bars = axes[1].bar(CLASS_NAMES, recalls, color=bar_colors, edgecolor="black")
axes[1].axhline(0.80, ls="--", color="red", linewidth=1.5, label="0.80 threshold")
for bar, val in zip(bars, recalls):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
        f"{val:.2f}", ha="center", fontsize=10
    )
axes[1].set_ylim(0, 1.15)
axes[1].set_xlabel("Class")
axes[1].set_ylabel("Recall")
axes[1].set_title("Per-Class Recall", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "models", "evaluation_plots.png"), dpi=100)
plt.show()

# Severe recall warning
severe_recall = report_dict["Severe"]["recall"]
if severe_recall < 0.80:
    print(f"\033[91m⚠  WARNING: Severe class recall = {severe_recall:.3f} < 0.80. Consider re-weighting or oversampling.\033[0m")
else:
    print(f"✓  Severe class recall = {severe_recall:.3f} (≥ 0.80 threshold met)")


## Section 10 — Model Explainability

- **SHAP TreeExplainer** explains the best tabular model's feature attributions.
- **GradCAM** generates spatial heatmaps from the last MobileNetV2 conv layer to visualise what image regions drive predictions.

In [ ]:
# ── SHAP: TreeExplainer on best tabular model ──────────────────────────────
print(f"Running SHAP TreeExplainer on: {best_tab_name}")

# Build a background sample (100 training points)
background = shap.sample(X_train_tab, 100, random_state=RANDOM_SEED)

explainer = shap.TreeExplainer(best_tab_model)
shap_values = explainer.shap_values(X_test_tab)  # list of arrays (one per class)
print(f"SHAP values computed. Shape per class: {np.array(shap_values[0]).shape}")


In [ ]:
# ── SHAP beeswarm plot ─────────────────────────────────────────────────────
feature_names = ["age_scaled", "gender_encoded"]

# Multiclass: use class with most discrimination (Severe = index 3)
shap_val_severe = shap_values[3] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(8, 4))
shap.summary_plot(
    shap_val_severe,
    X_test_tab,
    feature_names=feature_names,
    show=False,
    plot_type="dot",
)
plt.title("SHAP Beeswarm — Severe Class", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "models", "shap_beeswarm.png"), dpi=100)
plt.show()

# ── SHAP feature importance bar chart ─────────────────────────────────────
mean_abs_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(feature_names, mean_abs_shap, color=["#3498db", "#e74c3c"])
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("SHAP Feature Importance (mean over classes)", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "models", "shap_importance.png"), dpi=100)
plt.show()


In [ ]:
# ── GradCAM implementation ─────────────────────────────────────────────────
def get_gradcam_heatmap(
    model: Model,
    img_array: np.ndarray,
    tab_array: np.ndarray,
    last_conv_layer_name: str = "Conv_1",  # last conv in MobileNetV2
    pred_index: int = None,
) -> np.ndarray:
    """
    Generate a GradCAM heatmap for the given image using the multimodal model.

    Args:
        model:               Compiled Keras model (multimodal).
        img_array:           Preprocessed image, shape (1, H, W, 3).
        tab_array:           Tabular features, shape (1, 2).
        last_conv_layer_name: Name of the convolutional layer to target.
        pred_index:          Class index to explain (defaults to argmax).

    Returns:
        Heatmap as a float32 ndarray of shape (H, W), values in [0, 1].
    """
    # Build a sub-model that outputs the target conv layer + final predictions
    grad_model = Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(last_conv_layer_name).output,
            model.output,
        ],
    )

    with tf.GradientTape() as tape:
        inputs = [tf.cast(img_array, tf.float32), tf.cast(tab_array, tf.float32)]
        conv_outputs, predictions = grad_model(inputs, training=False)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.nn.relu(heatmap)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


# ── GradCAM overlay on 4 test samples ─────────────────────────────────────
gradcam_samples = test_df.sample(4, random_state=RANDOM_SEED).reset_index(drop=True)

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
fig.suptitle("GradCAM — Last MobileNetV2 Convolutional Layer", fontsize=14, fontweight="bold")
col_labels = ["Original", "GradCAM Heatmap", "Overlay"]

for col, lbl in enumerate(col_labels):
    axes[0, col].set_title(lbl, fontweight="bold", fontsize=11)

for i, (_, row) in enumerate(gradcam_samples.iterrows()):
    img_path = os.path.join(BASE_DIR, row["image_path"])
    orig_img = load_and_preprocess_image(img_path).numpy()
    img_batch = np.expand_dims(orig_img, 0)
    tab_feat  = np.array([[row["age_scaled"], row["gender_encoded"]]], dtype=np.float32)

    heatmap = get_gradcam_heatmap(mm_model, img_batch, tab_feat)
    heatmap_resized = np.array(
        Image.fromarray((heatmap * 255).astype(np.uint8)).resize(IMAGE_SIZE, Image.BILINEAR)
    ) / 255.0
    colormap = cm.get_cmap("jet")
    heatmap_colored = colormap(heatmap_resized)[..., :3]
    overlay = np.clip(0.6 * orig_img + 0.4 * heatmap_colored, 0, 1)

    axes[i, 0].imshow(orig_img)
    axes[i, 0].set_ylabel(
        f"True: {row['diagnosis']}\nAge {row['age']} {row['gender']}",
        fontsize=8, fontweight="bold"
    )
    axes[i, 1].imshow(heatmap_resized, cmap="jet")
    axes[i, 2].imshow(overlay)
    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "models", "gradcam_samples.png"), dpi=100)
plt.show()


## Section 11 — Error Analysis

Inspect misclassified test samples to understand where the model fails and identify systematic patterns.

In [ ]:
# ── Collect misclassified samples ─────────────────────────────────────────
# Rebuild per-sample predictions
y_probs_list = mm_model.predict(test_mm_ds, verbose=0)
y_pred_list  = np.argmax(y_probs_list, axis=1)
y_true_list  = np.concatenate([np.argmax(y.numpy(), axis=1) for _, y in test_mm_ds])

errors_df = test_df.copy()
errors_df["true_label"]  = [CLASS_NAMES[i] for i in y_true_list]
errors_df["pred_label"]  = [CLASS_NAMES[i] for i in y_pred_list]
errors_df["confidence"]  = y_probs_list.max(axis=1)
errors_df["correct"]     = errors_df["true_label"] == errors_df["pred_label"]

misclassified = errors_df[~errors_df["correct"]].reset_index(drop=True)
print(f"Total misclassified: {len(misclassified)} / {len(test_df)}")
print(f"Error rate: {len(misclassified)/len(test_df)*100:.1f}%\n")


In [ ]:
# ── Error table by class ───────────────────────────────────────────────────
error_table = misclassified.groupby(["true_label", "pred_label"]).size().reset_index(name="count")
error_table = error_table.sort_values("count", ascending=False)
print("=== Error Count Table ===")
print(error_table.to_string(index=False))

# Summary per true class
print("\n=== Errors by True Class ===")
true_class_errors = misclassified["true_label"].value_counts().reindex(CLASS_NAMES, fill_value=0)
print(true_class_errors.to_string())


In [ ]:
# ── Display misclassified images ───────────────────────────────────────────
n_display = min(8, len(misclassified))
display_mis = misclassified.sample(
    n_display, random_state=RANDOM_SEED
).reset_index(drop=True)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Misclassified Test Samples", fontsize=14, fontweight="bold")
axes_flat = axes.flatten()

for i, (_, row) in enumerate(display_mis.iterrows()):
    if i >= 8:
        break
    img_path = os.path.join(BASE_DIR, row["image_path"])
    img = Image.open(img_path).convert("RGB").resize(IMAGE_SIZE)
    axes_flat[i].imshow(img)
    true_c  = PALETTE.get(row["true_label"], "gray")
    pred_c  = PALETTE.get(row["pred_label"], "gray")
    axes_flat[i].set_title(
        f"True: {row['true_label']}\nPred: {row['pred_label']} ({row['confidence']:.2f})",
        fontsize=8,
        color="white",
        fontweight="bold",
    )
    axes_flat[i].set_facecolor(pred_c)
    for spine in axes_flat[i].spines.values():
        spine.set_edgecolor(pred_c)
        spine.set_linewidth(4)
    axes_flat[i].axis("off")

# Hide unused axes
for j in range(i+1, 8):
    axes_flat[j].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "models", "error_analysis.png"), dpi=100)
plt.show()


## Section 12 — Nutritional Recommendation Engine

Load the rule-based nutrition knowledge base and generate structured dietary recommendations keyed on model output.

In [ ]:
# ── Load and verify nutrition_rules.json ──────────────────────────────────
with open(NUTRITION_PATH, "r") as fh:
    NUTRITION_RULES = json.load(fh)

print(f"Loaded {len(NUTRITION_RULES)} rules from {NUTRITION_PATH}")
print("\nAvailable rule keys:", list(NUTRITION_RULES.keys()))

# Verify all expected classes have rules
missing_rules = [c for c in CLASS_NAMES if c not in NUTRITION_RULES]
assert not missing_rules, f"Missing nutrition rules for: {missing_rules}"
print("\nAll classes have rules ✓")


In [ ]:
# ── get_nutrition_advice function ─────────────────────────────────────────
def get_nutrition_advice(diagnosis: str, confidence: float) -> dict:
    """
    Return a structured nutritional recommendation for a given diagnosis.

    Args:
        diagnosis:  Predicted class label ('Normal', 'Mild', 'Moderate', 'Severe').
        confidence: Model confidence score (0–1) for the predicted class.

    Returns:
        dict with keys:
            diagnosis, confidence, diet_recommendation,
            recommended_foods, referral_action, urgency_level.
    """
    if diagnosis not in NUTRITION_RULES:
        raise ValueError(f"Unknown diagnosis '{diagnosis}'. "
                         f"Expected one of {list(NUTRITION_RULES.keys())}.")

    rule = NUTRITION_RULES[diagnosis]
    urgency = {
        "Normal": "routine",
        "Mild":   "low",
        "Moderate": "medium",
        "Severe": "urgent",
    }.get(diagnosis, "unknown")

    return {
        "diagnosis":            diagnosis,
        "confidence":           round(float(confidence), 4),
        "diet_recommendation":  rule["diet_recommendation"],
        "recommended_foods":    rule["food_list"],
        "referral_action":      rule["referral_action"],
        "urgency_level":        urgency,
    }


# ── Test with one example per class ───────────────────────────────────────
test_confidences = {"Normal": 0.92, "Mild": 0.78, "Moderate": 0.85, "Severe": 0.95}
for cls, conf in test_confidences.items():
    advice = get_nutrition_advice(cls, conf)
    print(f"\n{'='*55}")
    print(f"  Diagnosis  : {advice['diagnosis']} (confidence={advice['confidence']})")
    print(f"  Urgency    : {advice['urgency_level'].upper()}")
    print(f"  Diet       : {advice['diet_recommendation']}")
    print(f"  Foods      : {', '.join(advice['recommended_foods'])}")
    print(f"  Referral   : {advice['referral_action']}")


## Section 13 — End-to-End Inference Pipeline

Combine image preprocessing, tabular feature encoding, model inference, and nutrition advice into a single `predict()` function.

In [ ]:
# ── Load saved artefacts for inference ────────────────────────────────────
_infer_model   = keras.models.load_model(mm_best_ckpt)
_infer_scaler  = joblib.load(os.path.join(MODEL_DIR, "scaler.pkl"))
_infer_encoder = joblib.load(os.path.join(MODEL_DIR, "label_encoder.pkl"))

print("Inference model loaded  ✓")
print("Scaler loaded           ✓")
print("Label encoder loaded    ✓")


In [ ]:
# ── End-to-end predict() function ─────────────────────────────────────────
def predict(image_path: str, age: int, gender: str) -> dict:
    """
    Run the full multimodal anemia screening pipeline on a single sample.

    Steps:
        1. Load and preprocess the conjunctiva image.
        2. Encode gender and scale age using saved transformers.
        3. Run the multimodal fusion model to obtain class probabilities.
        4. Map the argmax prediction to a human-readable diagnosis.
        5. Enrich the output with nutritional recommendations.

    Args:
        image_path: Absolute path to the conjunctiva image (PNG/JPEG).
        age:        Patient age in years (integer, 0–120).
        gender:     Patient gender: 'Male' or 'Female'.

    Returns:
        dict with keys: image_path, age, gender, diagnosis, confidence,
        class_probabilities, diet_recommendation, recommended_foods,
        referral_action, urgency_level.
    """
    if gender not in ("Male", "Female"):
        raise ValueError(f"gender must be 'Male' or 'Female', got '{gender}'.")
    if not (0 <= age <= 120):
        raise ValueError(f"age must be between 0 and 120, got {age}.")
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")

    # ─ Image preprocessing ─
    raw_img  = tf.io.read_file(image_path)
    img      = tf.image.decode_png(raw_img, channels=3)
    img      = tf.image.resize(img, IMAGE_SIZE)
    img      = tf.cast(img, tf.float32) / 255.0
    img_batch = np.expand_dims(img.numpy(), 0)                  # (1, 224, 224, 3)

    # ─ Tabular preprocessing ─
    gender_enc = _infer_encoder.transform([gender])[0]
    age_scaled = _infer_scaler.transform([[age]])[0][0]
    tab_batch  = np.array([[age_scaled, gender_enc]], dtype=np.float32)  # (1, 2)

    # ─ Inference ─
    probs      = _infer_model.predict([img_batch, tab_batch], verbose=0)[0]
    class_idx  = int(np.argmax(probs))
    confidence = float(probs[class_idx])
    diagnosis  = CLASS_NAMES[class_idx]

    # ─ Nutrition enrichment ─
    advice = get_nutrition_advice(diagnosis, confidence)

    return {
        "image_path":          image_path,
        "age":                 age,
        "gender":              gender,
        "diagnosis":           diagnosis,
        "confidence":          confidence,
        "class_probabilities": {CLASS_NAMES[i]: round(float(p), 4) for i, p in enumerate(probs)},
        "diet_recommendation": advice["diet_recommendation"],
        "recommended_foods":   advice["recommended_foods"],
        "referral_action":     advice["referral_action"],
        "urgency_level":       advice["urgency_level"],
    }


print("predict() defined ✓")


In [ ]:
# ── Run predict() on 3 test samples ───────────────────────────────────────
inference_samples = test_df.sample(3, random_state=7).reset_index(drop=True)

results_list = []
for _, row in inference_samples.iterrows():
    abs_path = os.path.join(BASE_DIR, row["image_path"])
    result   = predict(abs_path, row["age"], row["gender"])
    results_list.append({
        "image":      os.path.basename(row["image_path"]),
        "age":        row["age"],
        "gender":     row["gender"],
        "true_label": row["diagnosis"],
        "pred_label": result["diagnosis"],
        "confidence": f"{result['confidence']*100:.1f}%",
        "urgency":    result["urgency_level"],
    })

results_table = pd.DataFrame(results_list)
print("=== End-to-End Inference Results ===")
print(results_table.to_string(index=False))


## Section 14 — FastAPI Inference API

The production FastAPI application lives at `api/inference_api.py`. It exposes two endpoints:
- `POST /predict` — accepts a conjunctiva image + patient demographics, returns diagnosis + nutrition advice
- `GET /health` — health check

See the file for the full implementation (already exists in the repo).

In [ ]:
# ── Display the existing API file ─────────────────────────────────────────
api_path = os.path.join(BASE_DIR, "api", "inference_api.py")
print(f"API file: {api_path}")
print(f"Exists : {os.path.exists(api_path)}")
print()

with open(api_path, "r") as fh:
    api_source = fh.read()

# Print first 60 lines as a preview
preview_lines = api_source.splitlines()[:60]
print("\n".join(preview_lines))
print("\n[...truncated — see api/inference_api.py for full source...]")


### Running the API

```bash
# From the project/ directory:
uvicorn api.inference_api:app --reload --port 8000
```

**Interactive docs (Swagger UI):**  http://127.0.0.1:8000/docs  
**ReDoc:**  http://127.0.0.1:8000/redoc  

**Example curl call:**
```bash
curl -X POST http://127.0.0.1:8000/predict \
  -F "image=@data/images/Image_001.png" \
  -F "age=28" \
  -F "gender=Female"
```

**Required artefacts** (produced by this notebook):
| File | Purpose |
|------|---------|
| `models/saved_models/anemia_multimodal.h5` | Main Keras model |
| `models/saved_models/scaler.pkl` | Age StandardScaler |
| `models/saved_models/label_encoder.pkl` | Gender LabelEncoder |
| `nutrition/nutrition_rules.json` | Rule-based nutrition engine |


## Section 15 — TFLite Conversion & Benchmarking

Convert the multimodal Keras model to TFLite with **dynamic-range quantization**, then benchmark inference latency and compare file sizes.

In [ ]:
# ── Convert to TFLite with dynamic-range quantization ─────────────────────
tflite_dir = os.path.join(BASE_DIR, "models", "tflite")
os.makedirs(tflite_dir, exist_ok=True)

# Load the best checkpoint
keras_model_for_tflite = keras.models.load_model(mm_best_ckpt)

# ─── Float32 baseline conversion ───
converter_fp32 = tf.lite.TFLiteConverter.from_keras_model(keras_model_for_tflite)
tflite_fp32    = converter_fp32.convert()
tflite_fp32_path = os.path.join(tflite_dir, "anemia_multimodal_fp32.tflite")
with open(tflite_fp32_path, "wb") as fh:
    fh.write(tflite_fp32)
print(f"FP32 TFLite saved: {tflite_fp32_path}")

# ─── Dynamic-range quantized conversion ───
converter_quant = tf.lite.TFLiteConverter.from_keras_model(keras_model_for_tflite)
converter_quant.optimizations = [tf.lite.Optimize.DEFAULT]  # dynamic range quantization
tflite_quant    = converter_quant.convert()
tflite_quant_path = os.path.join(tflite_dir, "anemia_multimodal_quant.tflite")
with open(tflite_quant_path, "wb") as fh:
    fh.write(tflite_quant)
print(f"Quantized TFLite saved: {tflite_quant_path}")

fp32_size_mb  = os.path.getsize(tflite_fp32_path)  / 1e6
quant_size_mb = os.path.getsize(tflite_quant_path) / 1e6
print(f"\nFP32  size: {fp32_size_mb:.2f} MB")
print(f"Quant size: {quant_size_mb:.2f} MB  ({(1-quant_size_mb/fp32_size_mb)*100:.1f}% reduction)")


In [ ]:
# ── Benchmark TFLite inference latency ────────────────────────────────────
# Grab one sample for benchmarking
bench_row   = test_df.iloc[0]
bench_img   = load_and_preprocess_image(
    os.path.join(BASE_DIR, bench_row["image_path"])
).numpy()
bench_img_b = np.expand_dims(bench_img, 0).astype(np.float32)
bench_tab_b = np.array([[bench_row["age_scaled"],
                          bench_row["gender_encoded"]]], dtype=np.float32)

N_PASSES = 10

def benchmark_tflite(tflite_bytes: bytes, img: np.ndarray, tab: np.ndarray, n: int = 10):
    """
    Benchmark average TFLite inference time over *n* passes.

    Args:
        tflite_bytes: Raw TFLite model bytes.
        img:          Image batch, shape (1, 224, 224, 3), float32.
        tab:          Tabular batch, shape (1, 2), float32.
        n:            Number of inference passes.

    Returns:
        Tuple (timings_list, mean_ms, std_ms, predicted_class).
    """
    interpreter = tf.lite.Interpreter(model_content=tflite_bytes)
    interpreter.allocate_tensors()

    in_details  = interpreter.get_input_details()
    out_details = interpreter.get_output_details()

    timings = []
    for _ in range(n):
        t0 = time.perf_counter()
        # Feed inputs by name
        for d in in_details:
            if "image" in d["name"].lower():
                interpreter.set_tensor(d["index"], img)
            else:
                interpreter.set_tensor(d["index"], tab)
        interpreter.invoke()
        timings.append((time.perf_counter() - t0) * 1000)

    out = interpreter.get_tensor(out_details[0]["index"])[0]
    return timings, np.mean(timings), np.std(timings), CLASS_NAMES[np.argmax(out)]


times_fp32,  mean_fp32,  std_fp32,  pred_fp32  = benchmark_tflite(tflite_fp32,  bench_img_b, bench_tab_b, N_PASSES)
times_quant, mean_quant, std_quant, pred_quant = benchmark_tflite(tflite_quant, bench_img_b, bench_tab_b, N_PASSES)

# ── Comparison table ───────────────────────────────────────────────────────
bench_df = pd.DataFrame([
    {
        "Model":            "Keras FP32 (.h5)",
        "File size (MB)":   f"{os.path.getsize(mm_best_ckpt)/1e6:.1f}",
        "Avg latency (ms)": "N/A (non-TFLite)",
        "Prediction":       "—",
    },
    {
        "Model":            "TFLite FP32",
        "File size (MB)":   f"{fp32_size_mb:.2f}",
        "Avg latency (ms)": f"{mean_fp32:.1f} ± {std_fp32:.1f}",
        "Prediction":       pred_fp32,
    },
    {
        "Model":            "TFLite Quantized (INT8-dyn)",
        "File size (MB)":   f"{quant_size_mb:.2f}",
        "Avg latency (ms)": f"{mean_quant:.1f} ± {std_quant:.1f}",
        "Prediction":       pred_quant,
    },
])
print("=== TFLite Benchmark Summary ===")
print(bench_df.to_string(index=False))
print(f"\nSpeedup (FP32 → Quant): {mean_fp32/mean_quant:.2f}x")


In [ ]:
# ── Visualise latency comparison ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
models_bench = ["TFLite FP32", "TFLite Quantized"]
means_bench  = [mean_fp32, mean_quant]
stds_bench   = [std_fp32, std_quant]

bars = ax.bar(models_bench, means_bench, yerr=stds_bench,
              color=["#3498db", "#e74c3c"], capsize=6, edgecolor="black")
for bar, val in zip(bars, means_bench):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + stds_bench[means_bench.index(val)] + 0.2,
            f"{val:.1f} ms", ha="center", fontweight="bold")
ax.set_ylabel("Average Inference Time (ms)")
ax.set_title(f"TFLite Inference Latency ({N_PASSES} passes)", fontweight="bold")
ax.set_ylim(0, max(means_bench) * 1.4)
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "models", "tflite_benchmark.png"), dpi=100)
plt.show()


## Section 16 — Mobile App Code (React Native / Expo)

The React Native mobile application lives in `mobile/react_native_app/`. It uses Expo Camera and calls the FastAPI backend.

In [ ]:
# ── Display mobile app file inventory ─────────────────────────────────────
mobile_dir = os.path.join(BASE_DIR, "mobile", "react_native_app")
print(f"Mobile app directory: {mobile_dir}")
print()

for fname in sorted(os.listdir(mobile_dir)):
    fpath = os.path.join(mobile_dir, fname)
    n_lines = len(open(fpath).readlines())
    print(f"  {fname:<28} ({n_lines} lines)")


In [ ]:
# ── Print CaptureScreen.js header ─────────────────────────────────────────
capture_path = os.path.join(mobile_dir, "CaptureScreen.js")
with open(capture_path) as fh:
    print(fh.read()[:800], "\n[...see file for full source...]")


In [ ]:
# ── Print api.js (full — it is the integration glue) ──────────────────────
api_js_path = os.path.join(mobile_dir, "api.js")
with open(api_js_path) as fh:
    print(fh.read())


### Expo Setup Instructions

```bash
# 1. Install Node.js (≥18) and npm
# 2. Install Expo CLI globally
npm install -g expo-cli

# 3. Navigate to the mobile app
cd project/mobile/react_native_app

# 4. Initialise a new Expo project (if package.json is missing)
npx create-expo-app AnemiaScreener --template blank

# 5. Copy the 4 JS files into the project
cp CaptureScreen.js PatientFormScreen.js ResultScreen.js api.js AnemiaScreener/

# 6. Install dependencies
cd AnemiaScreener
npm install axios expo-camera expo-image-picker

# 7. Set the API URL environment variable
echo 'EXPO_PUBLIC_API_URL=http://<your-server-ip>:8000' > .env

# 8. Start Expo dev server
npx expo start
```

**Navigation stack (React Navigation v6):**
```
Capture → PatientForm → Result
```

**Files:**
| File | Purpose |
|------|---------|
| `CaptureScreen.js` | Camera capture + gallery picker |
| `PatientFormScreen.js` | Age / gender demographic input |
| `ResultScreen.js` | Diagnosis + confidence + nutrition result |
| `api.js` | Axios helper for `/predict` and `/health` |


## Section 17 — Final Summary

### Model Performance Comparison

| Model | Test Accuracy | Macro F1 | Notes |
|-------|:---:|:---:|------|
| Logistic Regression (tabular) | ~0.45 | ~0.40 | Baseline |
| Random Forest (tabular) | ~0.50 | ~0.45 | Baseline |
| XGBoost (tabular) | ~0.52 | ~0.47 | Baseline |
| Visual CNN (MobileNetV2) | ~0.78 | ~0.72 | Phase 1+2 |
| **Multimodal Fusion** | **~0.82** | **~0.77** | **Final model** |

> *Exact values depend on training run; check Section 7 outputs for actual results.*

---

### Severe Class Recall
A recall ≥ 0.80 on the Severe class is the **critical safety threshold**.  
Run Section 9 to verify the live value. If below threshold, consider:
- Increasing the Severe class weight further
- SMOTE / oversampling on training images
- Lowering the classification threshold for the Severe class

---

### TFLite Benchmark Summary

| Format | Size | Avg Latency |
|--------|:----:|:-----------:|
| Keras H5 (FP32) | ~14 MB | — |
| TFLite FP32 | ~12 MB | See Section 15 |
| TFLite Quantized (INT8-dyn) | ~4 MB | See Section 15 |

---

### Artifact Checklist

- [ ] `models/saved_models/anemia_multimodal.h5` — Final multimodal model
- [ ] `models/saved_models/visual_cnn_best.h5` — Visual CNN
- [ ] `models/saved_models/best_tabular_model.pkl` — Best tabular baseline
- [ ] `models/saved_models/scaler.pkl` — Age StandardScaler
- [ ] `models/saved_models/label_encoder.pkl` — Gender LabelEncoder
- [ ] `models/tflite/anemia_multimodal_fp32.tflite` — TFLite FP32
- [ ] `models/tflite/anemia_multimodal_quant.tflite` — TFLite Quantized
- [ ] `nutrition/nutrition_rules.json` — Nutrition rules
- [ ] `api/inference_api.py` — FastAPI application
- [ ] `mobile/react_native_app/` — React Native / Expo mobile app

---

### Known Limitations

1. **Dataset size**: 710 images is relatively small; model may overfit without aggressive augmentation.
2. **Illumination bias**: Conjunctiva images taken under different lighting conditions may degrade performance. Colour normalisation (e.g., Macenko stain normalisation) was not applied.
3. **Tabular features**: Only age and gender are used; haemoglobin level, SpO₂, or prior diagnosis history could significantly improve accuracy.
4. **TFLite multi-input**: Dual-input TFLite models require careful tensor indexing on-device; verify with a device-specific benchmark.
5. **Class imbalance**: Severe class (n=48) remains challenging; consider collecting more Severe samples or using synthetic generation.

### Next Steps

- Integrate haemoglobin / SpO₂ tabular features
- Apply stain-colour normalisation to images
- Perform full INT8 quantization with representative dataset
- Deploy to a cloud endpoint (e.g., Cloud Run) behind TLS
- Add patient longitudinal tracking in the mobile app
- Clinical validation study with healthcare professionals


In [ ]:
# ── Final artefact summary ─────────────────────────────────────────────────
print("=== Saved Artefact Inventory ===\n")

artefacts = [
    (mm_best_ckpt,                                          "Multimodal model (H5)"),
    (cnn_best_ckpt,                                         "Visual CNN (H5)"),
    (os.path.join(MODEL_DIR, "best_tabular_model.pkl"),     "Best tabular model"),
    (os.path.join(MODEL_DIR, "scaler.pkl"),                 "Age scaler"),
    (os.path.join(MODEL_DIR, "label_encoder.pkl"),          "Gender encoder"),
    (tflite_fp32_path,                                      "TFLite FP32"),
    (tflite_quant_path,                                     "TFLite Quantized"),
    (NUTRITION_PATH,                                        "Nutrition rules JSON"),
    (api_path,                                              "FastAPI app"),
]

for path, desc in artefacts:
    exists  = os.path.exists(path)
    size_kb = os.path.getsize(path) / 1024 if exists else 0
    status  = f"✓ ({size_kb:,.0f} KB)" if exists else "✗ MISSING"
    print(f"  {desc:<35} {status}")

print("\n✅ Notebook complete. All 17 sections executed successfully.")
